
# 013 Network and Localization Analysis for MSAA Archetypes

This notebook follows the decoding/postprocessing notebooks and asks whether decoding-informative archetypes are distributed network-level motifs or increasingly localized/node-like features.

Main intended comparisons:

```python
spatial_across_K = [50, 88, 300, 700]
temporal_across_K = [14, 18, 60, 81]
```

For spatial AA, the node-level vector is taken from `S[k, :]`. For temporal AA, the node-level vector is taken from `sXC[:, k]`.


In [ ]:

# ============================================================
# SETTINGS
# ============================================================

FIT_SCOPE = "across"
MSAA_RESULTS_DIR = "."
DECODING_DIR_TEMPLATE = "msaa_condrank_decoding_outputs_{analysis_type}_{fit_scope}"

K_VALUES_BY_ANALYSIS = {
    "spatial": [
        5, 10, 14, 21, 35,
        46, 48, 50, 56,
        70, 88, 100, 300, 700
    ],

    "temporal": [
        2, 5, 6, 9, 15,
        20, 20, 23, 23,
        30, 38, 45, 129, 300
    ],
}

CONDITIONS = ["intact", "word", "rest"]
TOP_N_ARCHETYPES = 10
USE_TOP_DECODING_ARCHETYPES = True

# Network labels: define `network_labels` before running, or set one of these paths.
NETWORK_LABELS_NPY = None
NETWORK_LABELS_CSV = None
NETWORK_LABELS_VARIABLE = "network_labels"
NETWORK_NAME_MAP = {
    1: "Visual", 2: "SomMot", 3: "DorsAttn", 4: "VentAttn",
    5: "Limbic", 6: "FrontPar", 7: "Default",
}

# Optional coordinates: define `centers` before running, or set path.
NODE_COORDS_NPY = None
NODE_COORDS_VARIABLE = "centers"

N_PERMUTATIONS = 500
RANDOM_SEED = 0
WEIGHT_MODE = "abs"  # "abs", "positive", or "negative"

FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
FIG_NOTEBOOK_DIR = "013_network_localization_analysis"
SAVE_FIGS = True
FIG_FORMAT = "pdf"
DPI = 300
FIGSIZE = (8.5, 5)


# Atlas/posterior paths used in previous postprocessing notebooks
SCHAEFER_TXT = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order.txt"
SCHAEFER_NII = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm.nii.gz"
POSTERIOR_MAT = "data/pieman/raw/pieman_posterior_K700.mat"


In [ ]:

# ============================================================
# IMPORTS AND FIGURE HELPERS
# ============================================================

%matplotlib inline
from pathlib import Path
import os
import warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm, colors as mcolors
from matplotlib.colors import to_rgba, ListedColormap
from scipy.io import loadmat

try:
    from nilearn import plotting as niplot
    from nilearn.input_data import NiftiMasker
    import nibabel as nib
    NILEARN_AVAILABLE = True
except Exception:
    NILEARN_AVAILABLE = False

FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

_fig_counter = 0
def _safe_name(name):
    name = str(name).replace(" ", "_").replace("/", "-").replace("|", "_")
    name = "".join(ch for ch in name if ch.isalnum() or ch in ["_", "-", "."])
    return name[:180] if name else "figure"

def save_current_fig(name):
    global _fig_counter
    if not SAVE_FIGS:
        return None
    _fig_counter += 1
    out = FIG_DIR / f"{_fig_counter:03d}_{_safe_name(name)}.{FIG_FORMAT}"
    plt.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

print("Figure directory:", FIG_DIR)


In [ ]:

# ============================================================
# BASIC HELPERS
# ============================================================

def to_float_array(x):
    return np.asarray(x, dtype=float)

def normalize_abs_mass(w, eps=1e-12):
    w = np.asarray(w, dtype=float)
    return np.abs(w) / (np.sum(np.abs(w)) + eps)

def gini_coefficient(x, eps=1e-12):
    x = np.sort(np.abs(np.asarray(x, dtype=float)))
    if np.allclose(x.sum(), 0):
        return np.nan
    n = len(x)
    cumx = np.cumsum(x)
    return (n + 1 - 2 * np.sum(cumx) / (cumx[-1] + eps)) / n

def entropy_normalized(x, eps=1e-12):
    p = normalize_abs_mass(x, eps=eps)
    h = -np.sum(p * np.log(p + eps))
    return h / np.log(len(p))

def effective_n_nodes(x, eps=1e-12):
    p = normalize_abs_mass(x, eps=eps)
    return 1.0 / np.sum(p**2)

def participation_ratio(x, eps=1e-12):
    return effective_n_nodes(x, eps=eps) / len(x)

def weight_vector_by_mode(w, mode="abs"):
    w = np.asarray(w, dtype=float)
    if mode == "abs":
        return np.abs(w)
    if mode == "positive":
        return np.where(w > 0, w, 0.0)
    if mode == "negative":
        return np.where(w < 0, -w, 0.0)
    raise ValueError("WEIGHT_MODE must be 'abs', 'positive', or 'negative'.")


In [ ]:
# ============================================================
# LOAD NETWORK LABELS USING SAME METHOD AS PREVIOUS NOTEBOOKS
# ============================================================

posterior = loadmat(POSTERIOR_MAT)
centers = to_float_array(posterior['posterior']['centers'][0][0][0][0][0])
widths = to_float_array(list(posterior['posterior']['widths'][0][0][0][0][0][:, 0].T)).ravel()

lookup_table = {
    'Vis': 'Visual',
    'SomMot': 'Somatomotor',
    'DorsAttn': 'Dorsal attention',
    'SalVentAttn': 'Ventral attention',
    'Limbic': 'Limbic',
    'Cont': 'Frontoparietal',
    'Default': 'Default mode',
}

network_colors = {
    'Visual': '#D7DF23',
    'Somatomotor': '#39B54A',
    'Dorsal attention': '#00A79D',
    'Ventral attention': '#27AAE1',
    'Limbic': '#1C75BC',
    'Frontoparietal': '#92278F',
    'Default mode': '#EE2A7B',
}

network_codes = {k: i + 1 for i, k in enumerate(lookup_table.values())}
NETWORK_NAME_MAP = {v: k for k, v in network_codes.items()}

def nii2cmu(nifti_file, mask_file=None):
    def fullfact(dims):
        vals = np.asmatrix(range(1, dims[0] + 1)).T
        if len(dims) == 1:
            return vals
        aftervals = np.asmatrix(fullfact(dims[1:]))
        inds = np.asmatrix(np.zeros((np.prod(dims), len(dims))))
        row = 0
        for i in range(aftervals.shape[0]):
            inds[row:(row + len(vals)), 0] = vals
            inds[row:(row + len(vals)), 1:] = np.tile(aftervals[i, :], (len(vals), 1))
            row += len(vals)
        return inds

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        img = nib.load(nifti_file) if type(nifti_file) == str else nifti_file
        mask = NiftiMasker(mask_strategy='background')
        mask.fit(nifti_file if mask_file is None else mask_file)

    S = img.get_sform()
    Y = np.float32(mask.transform(nifti_file)).copy()

    vmask = np.nonzero(
        np.array(
            np.reshape(mask.mask_img_.dataobj, (1, np.prod(mask.mask_img_.shape)), order='C')
        )
    )[1]

    vox_coords = fullfact(img.shape[0:3])[vmask, ::-1] - 1
    R = np.array(np.dot(vox_coords, S[0:3, 0:3])) + S[:3, 3]

    return {'Y': Y, 'R': R}

def rbf(R, center, width):
    # EXACT version used in the earlier working notebooks / user-provided code.
    # Do not replace with the conventional 2*sigma^2 form because `width`
    # is already encoded in the posterior object in the CMU workflow.
    return np.exp(-np.sum((R - center) ** 2, axis=1) / width)

def node_labels(centers, widths, networks_cmu):
    labels = []

    for c, w in zip(centers, widths):
        r = rbf(networks_cmu['R'], c, w)
        label_weights = [
            sum(r[networks_cmu['Y'].ravel() == i])
            for i in range(1, len(network_codes) + 1)
        ]
        labels.append(np.argmax(label_weights) + 1)

    return pd.DataFrame({
        'code': labels,
        'Network': [list(lookup_table.values())[i - 1] for i in labels],
    })

if not NILEARN_AVAILABLE:
    raise ImportError("nilearn/nibabel are required for the same network-label workflow used in previous notebooks.")

key = pd.read_csv(
    SCHAEFER_TXT,
    sep='\t',
    header=None,
    names=['id', 'name', 'x', 'y', 'z', 't']
).drop('t', axis=1)

key['network'] = key['name'].apply(lambda x: lookup_table[x.split('_')[2]])
key['code'] = key['network'].apply(lambda x: network_codes[x])
key.set_index('id', inplace=True)
key.loc[0, 'code'] = 0

networks_cmu = nii2cmu(SCHAEFER_NII)
networks_cmu['Y'] = np.atleast_2d(
    np.array([key.loc[i, 'code'] for i in networks_cmu['Y']]).astype(float)
)

node_code_df = node_labels(centers, widths, networks_cmu)

network_labels = node_code_df['code'].to_numpy()
node_coords = centers

unique_networks = sorted(pd.unique(network_labels))

print("Loaded network labels using the same method as previous notebooks.")
print("Number of nodes:", len(network_labels))
print("Centers shape:", centers.shape)
print("Networks:", [(i, NETWORK_NAME_MAP.get(i, str(i))) for i in unique_networks])

def network_name(label):
    try:
        key = int(label)
    except Exception:
        key = label
    return NETWORK_NAME_MAP.get(key, str(label))


In [ ]:

# ============================================================
# FIND / LOAD MSAA AND DECODING OUTPUTS
# ============================================================

def find_msaa_npz(analysis_type, fit_scope, K, base_dir=MSAA_RESULTS_DIR):
    base_dir = Path(base_dir)
    candidates = []
    for p in base_dir.rglob("*.npz"):
        name = p.name.lower()
        if analysis_type.lower() in name and fit_scope.lower() in name and f"k{K}" in name:
            candidates.append(p)
    if len(candidates) == 0:
        for p in base_dir.rglob("*.npz"):
            name = p.name.lower()
            if analysis_type.lower() in name and f"k{K}" in name:
                candidates.append(p)
    if len(candidates) == 0:
        print(f"No npz found for {analysis_type} {fit_scope} K={K}")
        return None
    candidates = sorted(candidates, key=lambda p: (len(str(p)), str(p)))
    if len(candidates) > 1:
        print(f"Multiple candidates for {analysis_type} K={K}; using {candidates[0]}")
    return candidates[0]

def load_npz_as_dict(path):
    z = np.load(path, allow_pickle=True)
    out = {}
    for key in z.files:
        val = z[key]
        if val.shape == () and val.dtype == object:
            val = val.item()
        out[key] = val
    return out

def unpack_results_subj(d):
    for key in ["results_subj", "results", "subject_results", "subj_results"]:
        if key in d:
            obj = d[key]
            if isinstance(obj, list):
                return obj
            if isinstance(obj, np.ndarray) and obj.dtype == object:
                return list(obj)
    if "sXC" in d and "S" in d:
        return [{"sXC": d["sXC"], "S": d["S"]}]
    for val in d.values():
        if isinstance(val, np.ndarray) and val.dtype == object:
            maybe = list(val)
            if len(maybe) and isinstance(maybe[0], dict):
                return maybe
    raise ValueError("Could not unpack subject results from npz.")

def load_msaa_results(analysis_type, fit_scope, K):
    path = find_msaa_npz(analysis_type, fit_scope, K)
    if path is None:
        return None, None
    d = load_npz_as_dict(path)
    results_subj = unpack_results_subj(d)
    print(f"Loaded {analysis_type} {fit_scope} K={K}: {path}")
    return results_subj, path

def load_per_archetype_decoding(analysis_type, fit_scope):
    d = Path(DECODING_DIR_TEMPLATE.format(analysis_type=analysis_type, fit_scope=fit_scope))
    path = d / "per_archetype_mean_accuracy.csv"
    if not path.exists():
        print("Missing decoding:", path)
        return pd.DataFrame()
    df = pd.read_csv(path)
    rename = {}
    if "mean_accuracy" in df.columns and "mean" not in df.columns:
        rename["mean_accuracy"] = "mean"
    if "sem_accuracy" in df.columns and "sem" not in df.columns:
        rename["sem_accuracy"] = "sem"
    if "err" in df.columns and "sem" not in df.columns:
        rename["err"] = "sem"
    df = df.rename(columns=rename)
    if "analysis_type" in df.columns:
        df = df[df["analysis_type"].astype(str) == analysis_type].copy()
    if "fit_scope" in df.columns:
        df = df[df["fit_scope"].astype(str) == fit_scope].copy()
    df["K"] = df["K"].astype(int)
    df["archetype"] = df["archetype"].astype(int)
    df["condition"] = df["condition"].astype(str)
    print("Loaded decoding:", path, "rows:", len(df))
    return df

decoding_dfs = {a: load_per_archetype_decoding(a, FIT_SCOPE) for a in K_VALUES_BY_ANALYSIS}


In [ ]:

# ============================================================
# EXTRACT ARCHETYPE NODE VECTORS
# ============================================================

def get_spatial_vector(results_subj, analysis_type, k):
    vectors = []
    for sub in results_subj:
        sXC = to_float_array(sub["sXC"])
        S = to_float_array(sub["S"])
        if analysis_type == "spatial":
            v = S[k, :]
        elif analysis_type == "temporal":
            v = sXC[:, k]
        else:
            raise ValueError("analysis_type must be spatial or temporal")
        vectors.append(np.asarray(v, dtype=float))
    return np.nanmean(np.vstack(vectors), axis=0)

def get_top_archetypes(decoding_df, K, condition, top_n=TOP_N_ARCHETYPES):
    if decoding_df is None or len(decoding_df) == 0:
        return []
    sub = decoding_df[(decoding_df["K"] == int(K)) & (decoding_df["condition"] == condition)].copy()
    if len(sub) == 0:
        return []
    return sub.sort_values("mean", ascending=False)["archetype"].astype(int).head(top_n).tolist()


In [ ]:

# ============================================================
# NETWORK AND LOCALIZATION FUNCTIONS
# ============================================================

def network_mass_summary(vector, labels=network_labels, weight_mode=WEIGHT_MODE):
    v = np.asarray(vector, dtype=float)
    labels = np.asarray(labels)
    if len(v) != len(labels):
        raise ValueError(f"Vector length {len(v)} != labels length {len(labels)}")
    w = weight_vector_by_mode(v, mode=weight_mode)
    total = np.sum(w)
    rows = []
    for lab in sorted(pd.unique(labels)):
        idx = labels == lab
        mass = np.sum(w[idx])
        rows.append({
            "network_id": lab,
            "network": network_name(lab),
            "n_nodes": int(idx.sum()),
            "mass": float(mass),
            "mass_fraction": float(mass / total) if total > 0 else np.nan,
            "mean_abs_weight": float(np.mean(np.abs(v[idx]))),
            "mean_signed_weight": float(np.mean(v[idx])),
        })
    return pd.DataFrame(rows)

def network_enrichment_null(vector, labels=network_labels, n_perm=N_PERMUTATIONS, seed=RANDOM_SEED, weight_mode=WEIGHT_MODE):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    observed = network_mass_summary(vector, labels=labels, weight_mode=weight_mode)
    null_masses = {lab: [] for lab in observed["network_id"]}
    for _ in range(n_perm):
        perm_labels = rng.permutation(labels)
        null_df = network_mass_summary(vector, labels=perm_labels, weight_mode=weight_mode)
        for _, row in null_df.iterrows():
            null_masses[row["network_id"]].append(row["mass_fraction"])
    rows = []
    for _, row in observed.iterrows():
        lab = row["network_id"]
        null = np.asarray(null_masses[lab], dtype=float)
        obs = row["mass_fraction"]
        z = (obs - null.mean()) / (null.std() + 1e-12)
        p_hi = (np.sum(null >= obs) + 1) / (len(null) + 1)
        rows.append({**row.to_dict(), "null_mean": float(null.mean()), "null_std": float(null.std()), "enrichment_z": float(z), "p_enriched": float(p_hi)})
    return pd.DataFrame(rows)

def localization_summary(vector):
    return {
        "entropy_norm": float(entropy_normalized(vector)),
        "gini": float(gini_coefficient(vector)),
        "effective_n_nodes": float(effective_n_nodes(vector)),
        "participation_ratio": float(participation_ratio(vector)),
        "max_abs_weight": float(np.max(np.abs(vector))),
        "mean_abs_weight": float(np.mean(np.abs(vector))),
    }


In [ ]:

# ============================================================
# RUN ANALYSIS
# ============================================================

all_network_rows = []
all_localization_rows = []
all_selected_rows = []
loaded_cache = {}

for analysis_type, K_values in K_VALUES_BY_ANALYSIS.items():
    dec_df = decoding_dfs.get(analysis_type, pd.DataFrame())
    for K in K_values:
        results_subj, path = load_msaa_results(analysis_type, FIT_SCOPE, K)
        if results_subj is None:
            continue
        loaded_cache[(analysis_type, K)] = results_subj
        if USE_TOP_DECODING_ARCHETYPES:
            condition_to_archs = {cond: get_top_archetypes(dec_df, K, cond, TOP_N_ARCHETYPES) for cond in CONDITIONS}
        else:
            first = results_subj[0]
            n_arch = to_float_array(first["S"]).shape[0] if analysis_type == "spatial" else to_float_array(first["sXC"]).shape[1]
            condition_to_archs = {cond: list(range(n_arch)) for cond in CONDITIONS}
        for condition, archs in condition_to_archs.items():
            for rank, k in enumerate(archs, start=1):
                v = get_spatial_vector(results_subj, analysis_type, k)
                if len(v) != len(network_labels):
                    print(f"Skipping {analysis_type} K={K} a{k}: vector length {len(v)} != labels {len(network_labels)}")
                    continue
                dec_score = np.nan; dec_sem = np.nan
                if len(dec_df):
                    tmp = dec_df[(dec_df["K"] == K) & (dec_df["condition"] == condition) & (dec_df["archetype"] == k)]
                    if len(tmp):
                        dec_score = float(tmp.iloc[0]["mean"])
                        dec_sem = float(tmp.iloc[0]["sem"]) if "sem" in tmp.columns else np.nan
                info = {"analysis_type": analysis_type, "fit_scope": FIT_SCOPE, "K": int(K), "condition": condition, "archetype": int(k), "rank_within_condition": int(rank), "decoding_mean": dec_score, "decoding_sem": dec_sem, "source_npz": str(path)}
                all_selected_rows.append(info)
                all_localization_rows.append({**info, **localization_summary(v)})
                enr = network_enrichment_null(v, labels=network_labels, n_perm=N_PERMUTATIONS, seed=RANDOM_SEED + int(K) + int(k), weight_mode=WEIGHT_MODE)
                for _, row in enr.iterrows():
                    all_network_rows.append({**info, **row.to_dict()})

selected_df = pd.DataFrame(all_selected_rows)
localization_df = pd.DataFrame(all_localization_rows)
network_df = pd.DataFrame(all_network_rows)
print("Selected archetypes:", len(selected_df))
print("Localization rows:", len(localization_df))
print("Network rows:", len(network_df))
display(selected_df.head())
display(localization_df.head())
display(network_df.head())


In [ ]:

# ============================================================
# NETWORK MASS HEATMAPS FOR TOP ARCHETYPES
# ============================================================

def plot_network_heatmap(network_df, analysis_type, K, condition, top_n=TOP_N_ARCHETYPES):
    sub = network_df[(network_df["analysis_type"] == analysis_type) & (network_df["K"] == K) & (network_df["condition"] == condition) & (network_df["rank_within_condition"] <= top_n)].copy()
    if len(sub) == 0:
        print("No data for", analysis_type, K, condition)
        return
    sub["arch_label"] = sub.apply(lambda r: f"a{int(r['archetype'])} | r{int(r['rank_within_condition'])}", axis=1)
    mat = sub.pivot_table(index="arch_label", columns="network", values="mass_fraction", aggfunc="mean")
    fig, ax = plt.subplots(figsize=(9, max(3.5, 0.35 * len(mat))))
    im = ax.imshow(mat.to_numpy(), aspect="auto", cmap="viridis")
    ax.set_xticks(np.arange(mat.shape[1])); ax.set_xticklabels(mat.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(mat.shape[0])); ax.set_yticklabels(mat.index)
    ax.set_title(f"{analysis_type} | K={K} | {condition} | top-{top_n} network mass")
    ax.set_xlabel("Network"); ax.set_ylabel("Archetype")
    cbar = fig.colorbar(im, ax=ax); cbar.set_label("Mass fraction")
    plt.tight_layout(); save_current_fig(f"network_mass_heatmap_{analysis_type}_K{K}_{condition}"); plt.show(); plt.close()

for analysis_type, K_values in K_VALUES_BY_ANALYSIS.items():
    for K in K_values:
        for condition in ["intact", "word"]:
            plot_network_heatmap(network_df, analysis_type, K, condition, top_n=min(TOP_N_ARCHETYPES, 10))


In [ ]:
# ============================================================
# PATCHED BRAIN PLOT FUNCTION FOR OLDER NILEARN
# Replace only plot_top_decoded_brain_and_pie with this version
# ============================================================

def safe_plot_markers(values, coords, title):
    """
    Compatible with older nilearn versions.
    Tries increasingly simple plot_markers calls.
    """
    try:
        return niplot.plot_markers(
            values,
            coords,
            node_cmap="viridis",
            colorbar=True,
            display_mode="ortho",
            title=title,
        )
    except TypeError:
        try:
            return niplot.plot_markers(
                values,
                coords,
                node_cmap="viridis",
                display_mode="ortho",
                title=title,
            )
        except TypeError:
            try:
                return niplot.plot_markers(
                    values,
                    coords,
                    display_mode="ortho",
                    title=title,
                )
            except TypeError:
                return niplot.plot_markers(
                    values,
                    coords,
                    title=title,
                )


def plot_top_decoded_brain_and_pie(
    network_df,
    analysis_type,
    K,
    condition,
    rank_to_plot=1,
):
    ranked = get_ranked_archetypes(
        network_df,
        analysis_type,
        K,
        condition,
        top_n=TOP_N_ARCHETYPES,
    )

    if len(ranked) == 0:
        print("No ranked archetypes for", analysis_type, K, condition)
        return

    row = ranked[ranked["rank_within_condition"] == rank_to_plot]

    if len(row) == 0:
        print("No rank", rank_to_plot, "for", analysis_type, K, condition)
        return

    archetype = int(row.iloc[0]["archetype"])
    decoding_mean = float(row.iloc[0]["decoding_mean"])

    results_subj = loaded_cache.get((analysis_type, K), None)

    if results_subj is None:
        results_subj, _ = load_msaa_results(analysis_type, FIT_SCOPE, K)

    if results_subj is None:
        print("Could not load results for", analysis_type, K)
        return

    v = get_spatial_vector(results_subj, analysis_type, archetype)
    weights = weight_vector_by_mode(v, mode=WEIGHT_MODE)

    marker_values = weights / (np.nanmax(weights) + 1e-12)

    if NILEARN_AVAILABLE:
        safe_plot_markers(
            marker_values,
            node_coords,
            title=(
                f"{analysis_type} | K={K} | {condition}\n"
                f"top decoded a{archetype} | "
                f"rank {rank_to_plot} | dec={decoding_mean:.4f}"
            ),
        )

        save_current_fig(
            f"top_decoded_brain_weighted_"
            f"{analysis_type}_K{K}_{condition}_a{archetype}"
        )

        plt.show()
        plt.close()
    else:
        print("Nilearn unavailable; skipping brain plot.")

    pie_sub = network_df[
        (network_df["analysis_type"] == analysis_type) &
        (network_df["K"] == K) &
        (network_df["condition"] == condition) &
        (network_df["archetype"] == archetype)
    ].copy()

    if len(pie_sub) == 0:
        print("No network rows for pie.")
        return

    pie_sub = pie_sub.sort_values("mass_fraction", ascending=False)

    pie_colors = [
        network_colors.get(net, None)
        for net in pie_sub["network"]
    ]

    fig, ax = plt.subplots(figsize=(5.5, 5.5))

    ax.pie(
        pie_sub["mass_fraction"],
        labels=pie_sub["network"],
        autopct="%1.1f%%",
        startangle=90,
        colors=pie_colors,
        textprops={"fontsize": 9},
    )

    ax.set_title(
        f"{analysis_type} | K={K} | {condition}\n"
        f"top decoded a{archetype} raw network mass"
    )

    plt.tight_layout()

    save_current_fig(
        f"top_decoded_pie_mass_fraction_"
        f"{analysis_type}_K{K}_{condition}_a{archetype}"
    )

    plt.show()
    plt.close()

    enr = pie_sub.sort_values("enrichment_z", ascending=False)

    fig, ax = plt.subplots(figsize=(7, 4.5))

    ax.axhline(0, color="gray", linestyle="--", linewidth=1)

    ax.bar(enr["network"], enr["enrichment_z"])

    ax.set_ylabel("Network-size-controlled enrichment z")
    ax.set_xlabel("Network")

    ax.set_title(
        f"{analysis_type} | K={K} | {condition}\n"
        f"top decoded a{archetype} enrichment"
    )

    ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()

    save_current_fig(
        f"top_decoded_enrichment_bar_"
        f"{analysis_type}_K{K}_{condition}_a{archetype}"
    )

    plt.show()
    plt.close()

In [ ]:
colors = {
    1: network_colors["Visual"],
    2: network_colors["Somatomotor"],
    3: network_colors["Dorsal attention"],
    4: network_colors["Ventral attention"],
    5: network_colors["Limbic"],
    6: network_colors["Frontoparietal"],
    7: network_colors["Default mode"],
}

In [ ]:
# ============================================================
# CLEAN CELL:
# DECODING-RANKED NETWORK HEATMAPS
# + TOP 3 SPATIAL BRAIN PLOTS
# node color = network; opacity = spatial coefficient weight
# ============================================================

from matplotlib.colors import to_rgba

# ------------------------------------------------------------
# Settings for this cell
# ------------------------------------------------------------

TOP_BRAIN_RANKS = (1, 2, 3)


K_TO_PLOT = {
    "spatial": [
        5, 10, 14, 21, 35,
        46, 48, 50, 56,
        70, 88, 100, 300, 700
    ],

    "temporal": [
        2, 5, 6, 9, 15,
        20, 20, 23, 23,
        30, 38, 45, 129, 300
    ],
}


CONDITIONS_TO_PLOT = ["intact", "word"]

NETWORK_ORDER = [
    "Visual",
    "Somatomotor",
    "Dorsal attention",
    "Ventral attention",
    "Limbic",
    "Frontoparietal",
    "Default mode",
]


# ------------------------------------------------------------
# Ranking helper
# ------------------------------------------------------------

def get_ranked_archetypes(network_df, analysis_type, K, condition, top_n=TOP_N_ARCHETYPES):
    sub = network_df[
        (network_df["analysis_type"] == analysis_type) &
        (network_df["K"] == K) &
        (network_df["condition"] == condition) &
        (network_df["rank_within_condition"] <= top_n)
    ].copy()

    if len(sub) == 0:
        return pd.DataFrame()

    return (
        sub[["archetype", "rank_within_condition", "decoding_mean"]]
        .drop_duplicates()
        .sort_values("rank_within_condition")
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# Heatmap ordered by decoding rank
# ------------------------------------------------------------

def plot_network_heatmap_decoding_order(
    network_df,
    analysis_type,
    K,
    condition,
    top_n=TOP_N_ARCHETYPES,
    value_col="mass_fraction",
    cmap="viridis",
    center_zero=False,
):
    sub = network_df[
        (network_df["analysis_type"] == analysis_type) &
        (network_df["K"] == K) &
        (network_df["condition"] == condition) &
        (network_df["rank_within_condition"] <= top_n)
    ].copy()

    if len(sub) == 0:
        print("No heatmap data for", analysis_type, K, condition)
        return

    rank_df = get_ranked_archetypes(
        network_df,
        analysis_type,
        K,
        condition,
        top_n,
    )

    ordered_archs = rank_df["archetype"].tolist()

    mat = sub.pivot_table(
        index="archetype",
        columns="network",
        values=value_col,
        aggfunc="mean",
    )

    mat = mat.loc[[a for a in ordered_archs if a in mat.index]]
    mat = mat[[c for c in NETWORK_ORDER if c in mat.columns]]

    ylabels = []
    for a in mat.index:
        r = rank_df[rank_df["archetype"] == a].iloc[0]
        ylabels.append(
            f"a{int(a)} | r{int(r['rank_within_condition'])} | "
            f"dec={r['decoding_mean']:.4f}"
        )

    fig, ax = plt.subplots(figsize=(9, max(3.5, 0.38 * len(mat))))

    if center_zero:
        vmax = np.nanmax(np.abs(mat.to_numpy()))
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1
        im = ax.imshow(mat.to_numpy(), aspect="auto", cmap=cmap, vmin=-vmax, vmax=vmax)
    else:
        im = ax.imshow(mat.to_numpy(), aspect="auto", cmap=cmap)

    ax.set_xticks(np.arange(mat.shape[1]))
    ax.set_xticklabels(mat.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(mat.shape[0]))
    ax.set_yticklabels(ylabels)

    pretty_label = {
        "mass_fraction": "Raw mass fraction",
        "enrichment_z": "Network-size-controlled enrichment z",
    }.get(value_col, value_col)

    ax.set_title(
        f"{analysis_type} | K={K} | {condition} | top-{top_n}\n"
        f"{pretty_label}, ordered by decoding rank"
    )
    ax.set_xlabel("Network")
    ax.set_ylabel("Archetype ordered best-to-worst decoding")

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(pretty_label)

    plt.tight_layout()
    save_current_fig(
        f"network_heatmap_decoding_order_{value_col}_{analysis_type}_K{K}_{condition}"
    )
    plt.show()
    plt.close()


# ------------------------------------------------------------
# Spatial brain plot helpers
# ------------------------------------------------------------

def coefficient_to_alpha_local(weights, keep_mask, alpha_min=0.15, alpha_max=1.0):
    w = np.asarray(weights, dtype=float).ravel()
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)

    wk = w[keep_mask]

    if len(wk) == 0:
        return np.array([])

    wmin, wmax = wk.min(), wk.max()

    if np.isclose(wmax, wmin):
        return np.full(len(wk), alpha_max)

    scaled = (wk - wmin) / (wmax - wmin)
    return alpha_min + (alpha_max - alpha_min) * scaled


def get_spatial_coeff_vector_for_archetype(results_subj, archetype):
    """
    Spatial AA only:
    S[k, :] is the node-wise spatial coefficient vector.
    Average across subjects.
    """
    coeffs = np.stack(
        [np.asarray(sub["S"], dtype=float)[archetype, :] for sub in results_subj],
        axis=0,
    )
    return np.nan_to_num(coeffs.mean(axis=0), nan=0.0, posinf=0.0, neginf=0.0)


def plot_spatial_network_brain_previous_style(
    results_subj,
    K,
    condition,
    archetype,
    decoding_mean,
    display_mode="lyrz",
    node_size=10,
    thr_frac=0.30,
    alpha_min=0.15,
    alpha_max=1.0,
):
    """
    Spatial AA only.

    Node color = canonical network.
    Node opacity = spatial coefficient weight.
    """
    if not NILEARN_AVAILABLE:
        print("nilearn not available; skipping brain plot.")
        return None, None

    weights = get_spatial_coeff_vector_for_archetype(results_subj, archetype)

    wmax = np.max(weights)

    if wmax <= 0:
        print(f"Skipping K={K}, a{archetype}: non-positive weights")
        return None, None

    keep = weights >= (thr_frac * wmax)

    if keep.sum() == 0:
        print(f"Skipping K={K}, a{archetype}: no nodes pass threshold")
        return None, None

    centers_sel = centers[keep]

    node_codes_local = node_labels(centers, widths, networks_cmu)
    codes_sel = node_codes_local.loc[keep, "code"].to_numpy()

    alphas = coefficient_to_alpha_local(
        weights,
        keep,
        alpha_min=alpha_min,
        alpha_max=alpha_max,
    )

    node_colors = []
    for code, alpha in zip(codes_sel, alphas):
        base = to_rgba(colors[int(code)])
        node_colors.append((base[0], base[1], base[2], float(alpha)))

    display = niplot.plot_connectome(
        np.eye(centers_sel.shape[0]),
        centers_sel,
        node_size=node_size,
        node_color=node_colors,
        display_mode=display_mode,
        title=(
            f"spatial | K={K} | {condition}\n"
            f"ranked decoded archetype a{archetype} | dec={decoding_mean:.4f}"
        ),
    )

    save_current_fig(
        f"spatial_top_decoded_brain_K{K}_{condition}_a{archetype}"
    )

    plt.show()
    plt.close()
    plt.clf()

    return node_codes_local, keep


def plot_network_pie_for_selected_nodes_local(
    node_codes_local,
    keep_mask,
    title="Selected-node network composition",
):
    selected = node_codes_local.loc[keep_mask].copy()

    counts = (
        selected["Network"]
        .value_counts()
        .reindex(NETWORK_ORDER, fill_value=0)
    )

    fig, ax = plt.subplots(figsize=(4.8, 4.8))

    pie_colors = [network_colors.get(name, "gray") for name in counts.index]

    ax.pie(
        counts.values,
        labels=counts.index,
        colors=pie_colors,
        autopct=lambda p: f"{p:.1f}%" if p > 0 else "",
        startangle=90,
        counterclock=False,
        textprops={"fontsize": 9},
    )

    ax.set_title(title)

    plt.tight_layout()
    save_current_fig(title.replace(" ", "_").replace("|", "_"))
    plt.show()
    plt.close()

    return counts


# ------------------------------------------------------------
# Plot top 3 spatial brain maps
# ------------------------------------------------------------

def plot_top_spatial_brain_outputs(
    network_df,
    K,
    condition,
    ranks_to_plot=TOP_BRAIN_RANKS,
):
    ranked = get_ranked_archetypes(
        network_df,
        "spatial",
        K,
        condition,
        top_n=TOP_N_ARCHETYPES,
    )

    if len(ranked) == 0:
        print("No ranked spatial archetypes for", K, condition)
        return

    results_subj = loaded_cache.get(("spatial", K), None)

    if results_subj is None:
        results_subj, _ = load_msaa_results("spatial", FIT_SCOPE, K)

    if results_subj is None:
        print("Could not load spatial results for K=", K)
        return

    for rank_to_plot in ranks_to_plot:
        row = ranked[ranked["rank_within_condition"] == rank_to_plot]

        if len(row) == 0:
            print("No rank", rank_to_plot, "for spatial", K, condition)
            continue

        archetype = int(row.iloc[0]["archetype"])
        decoding_mean = float(row.iloc[0]["decoding_mean"])

        node_codes_local, keep = plot_spatial_network_brain_previous_style(
            results_subj=results_subj,
            K=K,
            condition=condition,
            archetype=archetype,
            decoding_mean=decoding_mean,
            display_mode="lyrz",
            node_size=10,
            thr_frac=0.30,
            alpha_min=0.15,
            alpha_max=1.0,
        )

        if node_codes_local is not None and keep is not None:
            plot_network_pie_for_selected_nodes_local(
                node_codes_local,
                keep,
                title=(
                    f"spatial | K={K} | {condition}\n"
                    f"rank {rank_to_plot} decoded a{archetype} selected-node composition"
                ),
            )


# ------------------------------------------------------------
# Wrapper
# ------------------------------------------------------------

def plot_all_network_summary_outputs(
    network_df,
    analysis_type,
    K,
    condition,
    top_n=TOP_N_ARCHETYPES,
):
    # These heatmaps are okay for both spatial and temporal because they use
    # the already-computed network_df summary.
    plot_network_heatmap_decoding_order(
        network_df,
        analysis_type=analysis_type,
        K=K,
        condition=condition,
        top_n=top_n,
        value_col="mass_fraction",
        cmap="viridis",
        center_zero=False,
    )

    plot_network_heatmap_decoding_order(
        network_df,
        analysis_type=analysis_type,
        K=K,
        condition=condition,
        top_n=top_n,
        value_col="enrichment_z",
        cmap="coolwarm",
        center_zero=True,
    )

    # Brain/node coefficient plots only make conceptual sense for spatial AA.
    if analysis_type == "spatial":
        plot_top_spatial_brain_outputs(
            network_df,
            K=K,
            condition=condition,
            ranks_to_plot=TOP_BRAIN_RANKS,
        )


# ============================================================
# RUN
# ============================================================
TOP_N_ARCHETYPES = 10
for analysis_type, K_values in K_TO_PLOT.items():
    for K in K_values:
        for condition in CONDITIONS_TO_PLOT:
            plot_all_network_summary_outputs(
                network_df,
                analysis_type=analysis_type,
                K=K,
                condition=condition,
                top_n=min(TOP_N_ARCHETYPES, 10),
            )

In [ ]:
# ============================================================
# SUMMARY ANALYSIS:
# Which networks are consistently represented among
# top-decoding archetypes across the best K regimes?
#
# Uses existing `network_df` from the notebook.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

SUMMARY_ANALYSIS_TYPE = "spatial"
SUMMARY_CONDITION = "intact"

K_SUMMARY = [5, 7, 10, 14, 21, 25, 35, 40, 42, 44, 45, 46, 48, 50, 52, 53, 54, 55, 56, 58, 60, 63, 65, 70, 75, 88, 100, 105, 126, 140, 175, 189, 200, 210, 252, 300, 400, 500, 600, 700]
TOP_N_LIST = [5, 7, 10]

NETWORK_ORDER = [
    "Visual",
    "Somatomotor",
    "Dorsal attention",
    "Ventral attention",
    "Limbic",
    "Frontoparietal",
    "Default mode",
]

# ------------------------------------------------------------
# Build summary table
# ------------------------------------------------------------

def summarize_top_network_mass(
    network_df,
    analysis_type="spatial",
    condition="intact",
    k_values=(14, 50, 70, 88),
    top_n=10,
):
    rows = []

    for K in k_values:
        sub = network_df[
            (network_df["analysis_type"] == analysis_type)
            & (network_df["condition"] == condition)
            & (network_df["K"] == K)
            & (network_df["rank_within_condition"] <= top_n)
        ].copy()

        if len(sub) == 0:
            print(f"No rows for {analysis_type}, {condition}, K={K}")
            continue

        for network in NETWORK_ORDER:
            s = sub[sub["network"] == network]

            rows.append({
                "analysis_type": analysis_type,
                "condition": condition,
                "K": K,
                "top_n": top_n,
                "network": network,
                "mean_mass_fraction": s["mass_fraction"].mean(),
                "sem_mass_fraction": s["mass_fraction"].std(ddof=1) / np.sqrt(max(len(s), 1)),
                "mean_enrichment_z": s["enrichment_z"].mean(),
                "sem_enrichment_z": s["enrichment_z"].std(ddof=1) / np.sqrt(max(len(s), 1)),
                "n_archetypes": s["archetype"].nunique(),
            })

    return pd.DataFrame(rows)


summary_dfs = []

for top_n in TOP_N_LIST:
    summary_dfs.append(
        summarize_top_network_mass(
            network_df,
            analysis_type=SUMMARY_ANALYSIS_TYPE,
            condition=SUMMARY_CONDITION,
            k_values=K_SUMMARY,
            top_n=top_n,
        )
    )

top_network_summary_df = pd.concat(summary_dfs, ignore_index=True)

display(top_network_summary_df.head())

# ------------------------------------------------------------
# Plot 1: mean raw mass fraction heatmap
# ------------------------------------------------------------

def plot_network_summary_heatmap(
    summary_df,
    value_col="mean_mass_fraction",
    top_n=10,
    title_prefix="Top-decoding archetypes",
    cmap="viridis",
):
    sub = summary_df[summary_df["top_n"] == top_n].copy()

    mat = sub.pivot(
        index="network",
        columns="K",
        values=value_col,
    ).reindex(NETWORK_ORDER)

    fig, ax = plt.subplots(figsize=(6.5, 4.4))

    im = ax.imshow(
        mat.values,
        aspect="auto",
        interpolation="nearest",
        cmap=cmap,
    )

    ax.set_xticks(np.arange(mat.shape[1]))
    ax.set_xticklabels(mat.columns)

    ax.set_yticks(np.arange(mat.shape[0]))
    ax.set_yticklabels(mat.index)

    ax.set_xlabel("Model order K")
    ax.set_ylabel("Network")

    label = "Mean mass fraction" if value_col == "mean_mass_fraction" else "Mean enrichment z-score"

    ax.set_title(
        f"{title_prefix}: top-{top_n}\n{SUMMARY_CONDITION}, {label}"
    )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(label)

    plt.tight_layout()

    save_current_fig(
        f"summary_network_{value_col}_top{top_n}_{SUMMARY_ANALYSIS_TYPE}_{SUMMARY_CONDITION}"
    )

    plt.show()
    plt.close()


for top_n in TOP_N_LIST:
    plot_network_summary_heatmap(
        top_network_summary_df,
        value_col="mean_mass_fraction",
        top_n=top_n,
        title_prefix="Network composition of top-decoding archetypes",
        cmap="viridis",
    )

# ------------------------------------------------------------
# Plot 2: enrichment heatmap relative to node-label null
# ------------------------------------------------------------

for top_n in TOP_N_LIST:
    plot_network_summary_heatmap(
        top_network_summary_df,
        value_col="mean_enrichment_z",
        top_n=top_n,
        title_prefix="Network enrichment of top-decoding archetypes",
        cmap="RdBu_r",
    )

# ------------------------------------------------------------
# Plot 3: bar plot showing network consistency across K
# ------------------------------------------------------------

def plot_network_consistency_bar(
    summary_df,
    top_n=10,
    value_col="mean_mass_fraction",
):
    sub = summary_df[summary_df["top_n"] == top_n].copy()

    agg = (
        sub.groupby("network")[value_col]
        .agg(["mean", "sem"])
        .reindex(NETWORK_ORDER)
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(7.2, 4.2))

    ax.bar(
        np.arange(len(agg)),
        agg["mean"],
        yerr=agg["sem"],
        capsize=3,
        edgecolor="black",
        linewidth=0.6,
    )

    ax.set_xticks(np.arange(len(agg)))
    ax.set_xticklabels(agg["network"], rotation=35, ha="right")

    ylabel = (
        "Mean mass fraction across K"
        if value_col == "mean_mass_fraction"
        else "Mean enrichment z-score across K"
    )

    ax.set_ylabel(ylabel)

    ax.set_title(
        f"Consistent network contributions across K={K_SUMMARY}\n"
        f"{SUMMARY_CONDITION}, top-{top_n} decoding archetypes"
    )

    ax.axhline(0, color="gray", linestyle="--", linewidth=1)

    plt.tight_layout()

    save_current_fig(
        f"network_consistency_bar_{value_col}_top{top_n}_{SUMMARY_ANALYSIS_TYPE}_{SUMMARY_CONDITION}"
    )

    plt.show()
    plt.close()


for top_n in TOP_N_LIST:
    plot_network_consistency_bar(
        top_network_summary_df,
        top_n=top_n,
        value_col="mean_mass_fraction",
    )

    plot_network_consistency_bar(
        top_network_summary_df,
        top_n=top_n,
        value_col="mean_enrichment_z",
    )

# ------------------------------------------------------------
# Print quick ranking table
# ------------------------------------------------------------

for top_n in TOP_N_LIST:
    print(f"\nTop-{top_n}: mean mass fraction across K={K_SUMMARY}")
    display(
        top_network_summary_df[top_network_summary_df["top_n"] == top_n]
        .groupby("network")["mean_mass_fraction"]
        .mean()
        .reindex(NETWORK_ORDER)
        .sort_values(ascending=False)
        .reset_index()
    )

    print(f"\nTop-{top_n}: mean enrichment z across K={K_SUMMARY}")
    display(
        top_network_summary_df[top_network_summary_df["top_n"] == top_n]
        .groupby("network")["mean_enrichment_z"]
        .mean()
        .reindex(NETWORK_ORDER)
        .sort_values(ascending=False)
        .reset_index()
    )

In [ ]:

# ============================================================
# AVERAGE NETWORK PROFILE OF TOP ARCHETYPES
# ============================================================

def plot_average_network_profile(network_df, analysis_type, K, condition, top_n=TOP_N_ARCHETYPES):
    sub = network_df[(network_df["analysis_type"] == analysis_type) & (network_df["K"] == K) & (network_df["condition"] == condition) & (network_df["rank_within_condition"] <= top_n)].copy()
    if len(sub) == 0:
        return
    avg = sub.groupby("network", as_index=False).agg(
        mass_fraction=("mass_fraction", "mean"),
        sem=("mass_fraction", lambda x: np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0),
        enrichment_z=("enrichment_z", "mean"),
    ).sort_values("mass_fraction", ascending=False)
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.bar(avg["network"], avg["mass_fraction"], yerr=avg["sem"], capsize=4)
    ax.set_ylabel("Mean mass fraction"); ax.set_xlabel("Network")
    ax.set_title(f"{analysis_type} | K={K} | {condition} | average top-{top_n} network profile")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout(); save_current_fig(f"avg_network_profile_{analysis_type}_K{K}_{condition}"); plt.show(); plt.close()
    display(avg)

for analysis_type, K_values in K_VALUES_BY_ANALYSIS.items():
    for K in K_values:
        for condition in ["intact", "word"]:
            plot_average_network_profile(network_df, analysis_type, K, condition)


In [ ]:

# ============================================================
# LOCALIZATION METRICS ACROSS K
# ============================================================

def plot_localization_metric(localization_df, metric, condition="intact"):
    sub = localization_df[localization_df["condition"] == condition].copy()
    if len(sub) == 0:
        return
    fig, ax = plt.subplots(figsize=FIGSIZE)
    for analysis_type in sorted(sub["analysis_type"].unique()):
        ss = sub[sub["analysis_type"] == analysis_type].groupby("K", as_index=False).agg(
            mean_metric=(metric, "mean"),
            sem_metric=(metric, lambda x: np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0),
        ).sort_values("K")
        ax.errorbar(ss["K"], ss["mean_metric"], yerr=ss["sem_metric"], marker="o", capsize=4, linewidth=2.2, label=analysis_type)
    ax.set_xlabel("K"); ax.set_ylabel(metric); ax.set_title(f"{metric} across K | condition={condition}")
    ax.legend(frameon=False); plt.tight_layout(); save_current_fig(f"localization_{metric}_{condition}"); plt.show(); plt.close()

for metric in ["gini", "entropy_norm", "effective_n_nodes", "participation_ratio"]:
    for condition in ["intact", "word"]:
        plot_localization_metric(localization_df, metric, condition=condition)


In [ ]:

# ============================================================
# SPATIAL MODERATE VS HIGH K COMPARISON
# ============================================================

SPATIAL_MODERATE_K = [50, 88]
SPATIAL_HIGH_K = [300, 700]
spatial_compare = localization_df[(localization_df["analysis_type"] == "spatial") & (localization_df["condition"].isin(["intact", "word"]))].copy()
spatial_compare["K_regime"] = np.where(spatial_compare["K"].isin(SPATIAL_MODERATE_K), "moderate", np.where(spatial_compare["K"].isin(SPATIAL_HIGH_K), "high", "other"))
spatial_compare = spatial_compare[spatial_compare["K_regime"].isin(["moderate", "high"])].copy()
summary = spatial_compare.groupby(["condition", "K_regime"], as_index=False).agg(
    gini_mean=("gini", "mean"), entropy_mean=("entropy_norm", "mean"), eff_nodes_mean=("effective_n_nodes", "mean"), participation_mean=("participation_ratio", "mean"), decoding_mean=("decoding_mean", "mean")
)
display(summary)

for metric in ["gini_mean", "entropy_mean", "eff_nodes_mean", "participation_mean", "decoding_mean"]:
    fig, ax = plt.subplots(figsize=(6, 4))
    for condition in ["intact", "word"]:
        sub = summary[summary["condition"] == condition]
        ax.plot(sub["K_regime"], sub[metric], marker="o", linewidth=2.2, label=condition)
    ax.set_ylabel(metric); ax.set_title(f"Spatial moderate vs high K: {metric}")
    ax.legend(frameon=False); plt.tight_layout(); save_current_fig(f"spatial_moderate_vs_high_{metric}"); plt.show(); plt.close()


In [ ]:

# ============================================================
# OPTIONAL NODE/BRAIN PLOTS FOR TOP INTACT ARCHETYPES
# ============================================================

def plot_node_values(vector, title, name):
    if node_coords is None:
        print("No node coordinates available; skipping node plot.")
        return
    v = np.asarray(vector, dtype=float)
    vmax = np.max(np.abs(v))
    if vmax == 0 or not np.isfinite(vmax): vmax = 1.0
    if NILEARN_AVAILABLE:
        try:
            disp = niplot.plot_markers(marker_values=v, marker_coords=node_coords, marker_size=25, cmap="coolwarm", display_mode="lyrz", colorbar=True, title=title)
            save_current_fig(name); plt.show()
            try: disp.close()
            except Exception: pass
            plt.close("all"); return
        except Exception as e:
            print("Nilearn plot failed, falling back to scatter:", repr(e))
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    sc0 = axes[0].scatter(node_coords[:, 0], node_coords[:, 1], c=v, cmap="coolwarm", vmin=-vmax, vmax=vmax)
    axes[0].set_xlabel("MNI x"); axes[0].set_ylabel("MNI y"); axes[0].set_title("x/y")
    sc1 = axes[1].scatter(node_coords[:, 0], node_coords[:, 2], c=v, cmap="coolwarm", vmin=-vmax, vmax=vmax)
    axes[1].set_xlabel("MNI x"); axes[1].set_ylabel("MNI z"); axes[1].set_title("x/z")
    fig.colorbar(sc1, ax=axes.ravel().tolist(), shrink=0.75)
    fig.suptitle(title); plt.tight_layout(); save_current_fig(name); plt.show(); plt.close()

for analysis_type, K_values in K_VALUES_BY_ANALYSIS.items():
    for K in K_values:
        results_subj = loaded_cache.get((analysis_type, K), None)
        if results_subj is None: continue
        top_row = selected_df[(selected_df["analysis_type"] == analysis_type) & (selected_df["K"] == K) & (selected_df["condition"] == "intact") & (selected_df["rank_within_condition"] == 1)]
        if len(top_row) == 0: continue
        k = int(top_row.iloc[0]["archetype"])
        v = get_spatial_vector(results_subj, analysis_type, k)
        plot_node_values(v, title=f"{analysis_type} K={K} archetype={k} top intact", name=f"nodeplot_{analysis_type}_K{K}_arch{k}_intact_top1")


In [ ]:

# ============================================================
# SAVE OUTPUT TABLES
# ============================================================

selected_path = FIG_DIR / f"selected_archetypes_{FIT_SCOPE}.csv"
loc_path = FIG_DIR / f"localization_summary_{FIT_SCOPE}.csv"
net_path = FIG_DIR / f"network_enrichment_{FIT_SCOPE}.csv"
selected_df.to_csv(selected_path, index=False)
localization_df.to_csv(loc_path, index=False)
network_df.to_csv(net_path, index=False)
print("Saved:", selected_path)
print("Saved:", loc_path)
print("Saved:", net_path)
display(selected_df.head(20))
display(localization_df.head(20))
display(network_df.head(20))


## Revised ratio-based and null-distribution analyses

The following cells revise notebook 13 so comparisons are made by normalized component ratio, restore the same brain-plotting style used in the postprocessing notebooks, and add null distributions.

In [ ]:
# ============================================================
# COMPONENT-RATIO HELPERS
# ============================================================
# Raw K is not directly comparable across spatial and temporal AA because the
# maximum meaningful component count differs across orientations.
#
# Spatial AA:   archetypes span the temporal dimension, coefficients are nodes.
#               Natural denominator = V = number of nodes, typically 700.
#
# Temporal AA:  archetypes span the spatial/node dimension, coefficients are time.
#               Natural denominator = T = number of timepoints, typically 300.
#
# Thus:
#     spatial ratio  = K / V
#     temporal ratio = K / T

SPATIAL_DENOMINATOR = len(network_labels)   # usually 700
TEMPORAL_DENOMINATOR = 300                  # update if your time series length differs

def component_ratio(K, analysis_type):
    if analysis_type == "spatial":
        return float(K) / float(SPATIAL_DENOMINATOR)
    if analysis_type == "temporal":
        return float(K) / float(TEMPORAL_DENOMINATOR)
    raise ValueError("analysis_type must be 'spatial' or 'temporal'.")

for df_name in ["selected_df", "localization_df", "network_df"]:
    if df_name in globals() and len(globals()[df_name]):
        df = globals()[df_name]
        df["component_ratio"] = df.apply(
            lambda r: component_ratio(int(r["K"]), str(r["analysis_type"])),
            axis=1
        )

print("Spatial denominator:", SPATIAL_DENOMINATOR)
print("Temporal denominator:", TEMPORAL_DENOMINATOR)

if "localization_df" in globals():
    display(localization_df[["analysis_type", "K", "component_ratio", "condition", "archetype", "decoding_mean"]].head())


In [ ]:
# ============================================================
# LOCALIZATION METRICS BY NORMALIZED COMPONENT RATIO
# ============================================================

def summarize_localization_by_ratio(localization_df, top_rank_max=TOP_N_ARCHETYPES):
    df = localization_df[
        localization_df["rank_within_condition"] <= top_rank_max
    ].copy()

    summary = (
        df.groupby(["analysis_type", "condition", "K", "component_ratio"], as_index=False)
        .agg(
            decoding_mean=("decoding_mean", "mean"),
            decoding_sem=("decoding_mean", lambda x: np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0),
            gini_mean=("gini", "mean"),
            gini_sem=("gini", lambda x: np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0),
            entropy_mean=("entropy_norm", "mean"),
            entropy_sem=("entropy_norm", lambda x: np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0),
            eff_nodes_mean=("effective_n_nodes", "mean"),
            eff_nodes_sem=("effective_n_nodes", lambda x: np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0),
            participation_mean=("participation_ratio", "mean"),
            participation_sem=("participation_ratio", lambda x: np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0),
        )
    )
    return summary

ratio_summary_df = summarize_localization_by_ratio(localization_df, top_rank_max=TOP_N_ARCHETYPES)
display(ratio_summary_df.head())

def plot_metric_by_ratio(summary_df, metric_base, condition="intact"):
    sub = summary_df[summary_df["condition"] == condition].copy()
    if len(sub) == 0:
        print("No rows for", condition)
        return

    fig, ax = plt.subplots(figsize=(7, 4.5))

    style = {
        "spatial": dict(linestyle="-", marker="o", label="Spatial AA"),
        "temporal": dict(linestyle="--", marker="o", label="Temporal AA"),
    }

    for analysis_type in ["spatial", "temporal"]:
        a = sub[sub["analysis_type"] == analysis_type].sort_values("component_ratio")
        if len(a) == 0:
            continue
        y = a[f"{metric_base}_mean"].to_numpy()
        err = a[f"{metric_base}_sem"].to_numpy() if f"{metric_base}_sem" in a else None
        ax.errorbar(
            a["component_ratio"],
            y,
            yerr=err,
            linewidth=2,
            capsize=3,
            **style[analysis_type]
        )

        # label K values lightly
        for _, r in a.iterrows():
            ax.text(
                r["component_ratio"],
                r[f"{metric_base}_mean"],
                f"K={int(r['K'])}",
                fontsize=7,
                ha="center",
                va="bottom"
            )

    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel(metric_base.replace("_", " "))
    ax.set_title(f"{condition}: {metric_base} by normalized component ratio")
    ax.legend(frameon=False)
    plt.tight_layout()
    save_current_fig(f"ratio_{condition}_{metric_base}")
    plt.show()
    plt.close()

for condition in ["intact", "word", "rest"]:
    for metric in ["decoding", "gini", "entropy", "eff_nodes", "participation"]:
        metric_base = "decoding" if metric == "decoding" else metric
        if metric_base == "decoding":
            # decoding_mean/decoding_sem columns exist
            plot_metric_by_ratio(ratio_summary_df, "decoding", condition=condition)
        elif metric_base == "gini":
            plot_metric_by_ratio(ratio_summary_df, "gini", condition=condition)
        elif metric_base == "entropy":
            plot_metric_by_ratio(ratio_summary_df, "entropy", condition=condition)
        elif metric_base == "eff_nodes":
            plot_metric_by_ratio(ratio_summary_df, "eff_nodes", condition=condition)
        elif metric_base == "participation":
            plot_metric_by_ratio(ratio_summary_df, "participation", condition=condition)


In [ ]:
colors = {
    1: network_colors["Visual"],
    2: network_colors["Somatomotor"],
    3: network_colors["Dorsal attention"],
    4: network_colors["Ventral attention"],
    5: network_colors["Limbic"],
    6: network_colors["Frontoparietal"],
    7: network_colors["Default mode"],
}

In [ ]:
# ============================================================
# BRAIN PLOTS USING THE SAME STYLE AS THE PREVIOUS NOTEBOOKS
# ============================================================

def coefficient_to_alpha(weights, keep_mask, alpha_min=0.15, alpha_max=1.0):
    w = np.asarray(weights, dtype=float).ravel()
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    wk = w[keep_mask]
    if len(wk) == 0:
        return np.array([])
    wmin, wmax = wk.min(), wk.max()
    if np.isclose(wmax, wmin):
        return np.full(len(wk), alpha_max)
    scaled = (wk - wmin) / (wmax - wmin)
    return alpha_min + (alpha_max - alpha_min) * scaled

def plot_spatialAA_network_colored_coeff_map(results_subj, k, K, title="", display_mode="lyrz",
                                             node_size=10, thr_frac=0.30, use_opacity=True,
                                             alpha_min=0.15, alpha_max=1.0):
    """
    Same spatial-AA coefficient brain plot style as previous notebooks.

    Spatial AA:
        sXC[:, k] = timecourse archetype
        S[k, :]   = node/spatial coefficients
    """
    if not NILEARN_AVAILABLE:
        print("nilearn not available; skipping.")
        return None, None, None

    coeffs = np.stack([np.asarray(sub["S"], dtype=float)[k, :] for sub in results_subj], axis=0)
    weights = np.nan_to_num(coeffs.mean(axis=0), nan=0.0, posinf=0.0, neginf=0.0)

    wmax = np.max(weights)
    if wmax <= 0 or not np.isfinite(wmax):
        print(f"Skipping spatial AA K={K}, archetype {k}: non-positive/invalid weights.")
        return None, None, None

    keep = weights >= (thr_frac * wmax)
    centers_sel = centers[keep]

    node_codes_local = node_labels(centers, widths, networks_cmu)
    codes_sel = node_codes_local.loc[keep, "code"].to_numpy()

    if use_opacity:
        alphas = coefficient_to_alpha(weights, keep, alpha_min=alpha_min, alpha_max=alpha_max)
        node_colors = []
        for code, alpha in zip(codes_sel, alphas):
            base = to_rgba(colors[int(code)])
            node_colors.append((base[0], base[1], base[2], float(alpha)))
    else:
        node_colors = [colors[int(i)] for i in codes_sel]

    disp = niplot.plot_connectome(
        np.eye(centers_sel.shape[0]),
        centers_sel,
        node_size=node_size,
        node_color=node_colors,
        display_mode=display_mode,
        title=title or f"Spatial AA | K={K} | archetype {k} | spatial coefficients",
    )

    save_current_fig(f"brain_spatialAA_K{K}_arch{k}_network_opacity{use_opacity}")
    plt.show()
    plt.close()

    try:
        disp.close()
    except Exception:
        pass

    return disp, node_codes_local, keep

def plot_temporalAA_signed_spatial_motif_map(results_subj, k, K, display_mode="lyrz"):
    """
    Same temporal-AA signed spatial motif map style as previous temporal notebooks.

    Temporal AA:
        sXC[:, k] = spatial archetype / spatial motif
        S[k, :]   = time-varying coefficient
    """
    if not NILEARN_AVAILABLE:
        print("nilearn not available; skipping.")
        return None

    vals = np.stack(
        [to_float_array(sub["sXC"])[:, k] for sub in results_subj],
        axis=0
    ).mean(axis=0)

    vals = np.nan_to_num(vals)

    vmax = np.max(np.abs(vals))
    if vmax == 0 or not np.isfinite(vmax):
        vmax = 1.0

    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
    cmap = cm.get_cmap("coolwarm")

    node_colors = [cmap(norm(v)) for v in vals]
    node_sizes = 8 + 18 * (np.abs(vals) / (vmax + 1e-8))

    disp = niplot.plot_connectome(
        np.eye(centers.shape[0]),
        centers,
        node_color=node_colors,
        node_size=node_sizes,
        display_mode=display_mode,
        title=f"Temporal AA | K={K} | archetype {k} | signed spatial motif",
    )

    save_current_fig(f"brain_temporalAA_K{K}_arch{k}_signed_spatial_motif")
    plt.show()
    plt.close()

    try:
        disp.close()
    except Exception:
        pass

    return disp

# Plot only a small, editable set of examples so the notebook does not produce hundreds of files.
PLOT_BRAIN_EXAMPLES = True
BRAINPLOT_CONDITIONS = ["intact"]
BRAINPLOT_TOP_RANKS = [1, 2, 3]
BRAINPLOT_K_BY_ANALYSIS = {
    "spatial": [5, 50, 700],
    "temporal": [5, 23, 300],
}

if PLOT_BRAIN_EXAMPLES:
    for analysis_type, K_list in BRAINPLOT_K_BY_ANALYSIS.items():
        dec_df = decoding_dfs.get(analysis_type, pd.DataFrame())
        for K in K_list:
            key = (analysis_type, K)
            if key in loaded_cache:
                results_subj = loaded_cache[key]
            else:
                results_subj, _ = load_msaa_results(analysis_type, FIT_SCOPE, K)

            if results_subj is None:
                continue

            for condition in BRAINPLOT_CONDITIONS:
                archs = get_top_archetypes(dec_df, K, condition, top_n=max(BRAINPLOT_TOP_RANKS))
                for rank_idx in BRAINPLOT_TOP_RANKS:
                    if rank_idx > len(archs):
                        continue
                    k = int(archs[rank_idx - 1])

                    if analysis_type == "spatial":
                        plot_spatialAA_network_colored_coeff_map(
                            results_subj,
                            k,
                            K,
                            title=f"Spatial AA | K={K} | {condition} | rank {rank_idx} | archetype {k}",
                            use_opacity=True,
                            thr_frac=0.30,
                        )
                    elif analysis_type == "temporal":
                        plot_temporalAA_signed_spatial_motif_map(
                            results_subj,
                            k,
                            K,
                        )


In [ ]:
# ============================================================
# NULL ANALYSES:
#   1) permutation null for network labels
#   2) random-archetype null for localization / network concentration
# ============================================================

# Note: network_enrichment_null() above already performs a node-label
# permutation null for each archetype and stores enrichment_z / p_enriched
# in network_df. This cell adds explicit visual summaries and a second null
# based on randomly selected archetypes.

def plot_enrichment_z_by_ratio(network_df, condition="intact"):
    sub = network_df[network_df["condition"] == condition].copy()
    if len(sub) == 0:
        print("No network enrichment rows for", condition)
        return

    # For each selected archetype, summarize the strongest absolute network enrichment.
    arch_enrich = (
        sub.groupby(["analysis_type", "K", "component_ratio", "archetype", "rank_within_condition"], as_index=False)
        .agg(max_abs_enrichment_z=("enrichment_z", lambda x: np.max(np.abs(x))))
    )

    summary = (
        arch_enrich.groupby(["analysis_type", "K", "component_ratio"], as_index=False)
        .agg(
            mean_max_abs_z=("max_abs_enrichment_z", "mean"),
            sem_max_abs_z=("max_abs_enrichment_z", lambda x: np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0),
        )
    )

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for analysis_type, ls in [("spatial", "-"), ("temporal", "--")]:
        a = summary[summary["analysis_type"] == analysis_type].sort_values("component_ratio")
        if len(a) == 0:
            continue
        ax.errorbar(
            a["component_ratio"],
            a["mean_max_abs_z"],
            yerr=a["sem_max_abs_z"],
            marker="o",
            linestyle=ls,
            linewidth=2,
            capsize=3,
            label=f"{analysis_type} AA",
        )
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel("Mean max |network enrichment z|")
    ax.set_title(f"{condition}: network enrichment versus node-label permutation null")
    ax.legend(frameon=False)
    plt.tight_layout()
    save_current_fig(f"network_label_permutation_null_{condition}")
    plt.show()
    plt.close()

for condition in ["intact", "word", "rest"]:
    plot_enrichment_z_by_ratio(network_df, condition=condition)


def get_n_archetypes_from_results(results_subj, analysis_type):
    first = results_subj[0]
    if analysis_type == "spatial":
        return to_float_array(first["S"]).shape[0]
    if analysis_type == "temporal":
        return to_float_array(first["sXC"]).shape[1]
    raise ValueError("analysis_type must be spatial or temporal.")

def random_archetype_null_for_K(analysis_type, K, condition, n_iter=500, top_n=TOP_N_ARCHETYPES, seed=0):
    """
    Compare the observed top-decoding archetype localization against random
    archetype sets drawn from the same fitted model and same K.
    """
    rng = np.random.default_rng(seed)

    results_subj = loaded_cache.get((analysis_type, K), None)
    if results_subj is None:
        results_subj, _ = load_msaa_results(analysis_type, FIT_SCOPE, K)

    if results_subj is None:
        return None

    n_arch = get_n_archetypes_from_results(results_subj, analysis_type)

    observed = localization_df[
        (localization_df["analysis_type"] == analysis_type) &
        (localization_df["K"] == K) &
        (localization_df["condition"] == condition) &
        (localization_df["rank_within_condition"] <= top_n)
    ].copy()

    if len(observed) == 0:
        return None

    obs_summary = {
        "gini": observed["gini"].mean(),
        "entropy_norm": observed["entropy_norm"].mean(),
        "effective_n_nodes": observed["effective_n_nodes"].mean(),
        "participation_ratio": observed["participation_ratio"].mean(),
        "decoding_mean": observed["decoding_mean"].mean(),
    }

    null_rows = []
    draw_n = min(top_n, n_arch)
    for i in range(n_iter):
        random_archs = rng.choice(np.arange(n_arch), size=draw_n, replace=False)
        vals = []
        for k in random_archs:
            v = get_spatial_vector(results_subj, analysis_type, int(k))
            vals.append(localization_summary(v))
        tmp = pd.DataFrame(vals)
        null_rows.append({
            "iter": i,
            "gini": tmp["gini"].mean(),
            "entropy_norm": tmp["entropy_norm"].mean(),
            "effective_n_nodes": tmp["effective_n_nodes"].mean(),
            "participation_ratio": tmp["participation_ratio"].mean(),
        })

    null = pd.DataFrame(null_rows)

    rows = []
    for metric, obs in obs_summary.items():
        if metric not in null.columns:
            continue
        x = null[metric].to_numpy(dtype=float)
        p_hi = (np.sum(x >= obs) + 1) / (len(x) + 1)
        p_lo = (np.sum(x <= obs) + 1) / (len(x) + 1)
        rows.append({
            "analysis_type": analysis_type,
            "K": K,
            "component_ratio": component_ratio(K, analysis_type),
            "condition": condition,
            "metric": metric,
            "observed": obs,
            "null_mean": float(np.mean(x)),
            "null_std": float(np.std(x)),
            "z": float((obs - np.mean(x)) / (np.std(x) + 1e-12)),
            "p_high": float(p_hi),
            "p_low": float(p_lo),
        })

    return pd.DataFrame(rows), null

RANDOM_ARCHETYPE_NULL_ITER = 500
random_null_results = []

# Keep this compact: intact and word are probably the most useful.
for analysis_type, K_values in K_VALUES_BY_ANALYSIS.items():
    for K in K_values:
        for condition in ["intact", "word"]:
            out = random_archetype_null_for_K(
                analysis_type,
                int(K),
                condition,
                n_iter=RANDOM_ARCHETYPE_NULL_ITER,
                top_n=TOP_N_ARCHETYPES,
                seed=RANDOM_SEED + int(K),
            )
            if out is None:
                continue
            summary_df, _ = out
            random_null_results.append(summary_df)

random_archetype_null_df = pd.concat(random_null_results, ignore_index=True) if random_null_results else pd.DataFrame()
display(random_archetype_null_df.head())

def plot_random_archetype_null_metric(null_df, metric="gini", condition="intact"):
    sub = null_df[(null_df["metric"] == metric) & (null_df["condition"] == condition)].copy()
    if len(sub) == 0:
        print("No null rows:", metric, condition)
        return

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for analysis_type, ls in [("spatial", "-"), ("temporal", "--")]:
        a = sub[sub["analysis_type"] == analysis_type].sort_values("component_ratio")
        if len(a) == 0:
            continue
        ax.plot(
            a["component_ratio"],
            a["observed"],
            marker="o",
            linestyle=ls,
            linewidth=2,
            label=f"{analysis_type} observed",
        )
        ax.plot(
            a["component_ratio"],
            a["null_mean"],
            linestyle=ls,
            linewidth=1,
            alpha=0.6,
            label=f"{analysis_type} random-archetype null",
        )
        ax.fill_between(
            a["component_ratio"],
            a["null_mean"] - 1.96 * a["null_std"],
            a["null_mean"] + 1.96 * a["null_std"],
            alpha=0.15,
        )

    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel(metric)
    ax.set_title(f"{condition}: observed top-decoding archetypes vs random archetypes")
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    save_current_fig(f"random_archetype_null_{condition}_{metric}")
    plt.show()
    plt.close()

for condition in ["intact", "word"]:
    for metric in ["gini", "entropy_norm", "effective_n_nodes", "participation_ratio"]:
        plot_random_archetype_null_metric(random_archetype_null_df, metric=metric, condition=condition)


## Refined strict random-archetype null and matched-ratio comparison

Run these cells after the original localization/network summaries have been computed.

In [ ]:

# ============================================================
# REFINED RANDOM-ARCHETYPE NULL
# ============================================================
# Only run random-archetype nulls when K > top_m.
# If K <= top_m, the "top-m" set is all/nearly all archetypes, so a random
# archetype set is not a meaningful comparison.

STRICT_RANDOM_NULL_TOP_M = TOP_N_ARCHETYPES
STRICT_RANDOM_NULL_ITER = 1000
STRICT_RANDOM_NULL_CONDITIONS = ["intact", "word"]

def get_n_archetypes_from_results(results_subj, analysis_type):
    first = results_subj[0]
    if analysis_type == "spatial":
        return to_float_array(first["S"]).shape[0]
    if analysis_type == "temporal":
        return to_float_array(first["sXC"]).shape[1]
    raise ValueError("analysis_type must be spatial or temporal.")

def random_archetype_null_strict(
    analysis_type,
    K,
    condition,
    top_m=STRICT_RANDOM_NULL_TOP_M,
    n_iter=STRICT_RANDOM_NULL_ITER,
    seed=0,
):
    results_subj = loaded_cache.get((analysis_type, K), None)
    if results_subj is None:
        results_subj, _ = load_msaa_results(analysis_type, FIT_SCOPE, K)

    if results_subj is None:
        return None, {"reason": "missing_fit", "analysis_type": analysis_type, "K": K, "condition": condition}

    n_arch = get_n_archetypes_from_results(results_subj, analysis_type)

    if n_arch <= top_m:
        return None, {
            "reason": "K_or_n_arch_not_greater_than_top_m",
            "analysis_type": analysis_type,
            "K": K,
            "condition": condition,
            "n_arch": n_arch,
            "top_m": top_m,
        }

    observed = localization_df[
        (localization_df["analysis_type"] == analysis_type) &
        (localization_df["K"] == K) &
        (localization_df["condition"] == condition) &
        (localization_df["rank_within_condition"] <= top_m)
    ].copy()

    if len(observed) == 0:
        return None, {"reason": "no_observed_top_archetypes", "analysis_type": analysis_type, "K": K, "condition": condition}

    obs_summary = {
        "gini": observed["gini"].mean(),
        "entropy_norm": observed["entropy_norm"].mean(),
        "effective_n_nodes": observed["effective_n_nodes"].mean(),
        "participation_ratio": observed["participation_ratio"].mean(),
    }

    rng = np.random.default_rng(seed)
    null_rows = []

    for i in range(n_iter):
        random_archs = rng.choice(np.arange(n_arch), size=top_m, replace=False)
        vals = []

        for k in random_archs:
            v = get_spatial_vector(results_subj, analysis_type, int(k))
            vals.append(localization_summary(v))

        tmp = pd.DataFrame(vals)
        null_rows.append({
            "iter": i,
            "gini": tmp["gini"].mean(),
            "entropy_norm": tmp["entropy_norm"].mean(),
            "effective_n_nodes": tmp["effective_n_nodes"].mean(),
            "participation_ratio": tmp["participation_ratio"].mean(),
        })

    null = pd.DataFrame(null_rows)

    rows = []
    for metric, obs in obs_summary.items():
        x = null[metric].to_numpy(dtype=float)
        rows.append({
            "analysis_type": analysis_type,
            "K": int(K),
            "component_ratio": component_ratio(K, analysis_type),
            "condition": condition,
            "top_m": int(top_m),
            "n_arch": int(n_arch),
            "metric": metric,
            "observed": float(obs),
            "null_mean": float(np.mean(x)),
            "null_std": float(np.std(x)),
            "z": float((obs - np.mean(x)) / (np.std(x) + 1e-12)),
            "p_high": float((np.sum(x >= obs) + 1) / (len(x) + 1)),
            "p_low": float((np.sum(x <= obs) + 1) / (len(x) + 1)),
        })

    return pd.DataFrame(rows), None


strict_random_null_rows = []
strict_random_null_skipped = []

for analysis_type, K_values in K_VALUES_BY_ANALYSIS.items():
    for K in K_values:
        for condition in STRICT_RANDOM_NULL_CONDITIONS:
            out, skipped = random_archetype_null_strict(
                analysis_type,
                int(K),
                condition,
                top_m=STRICT_RANDOM_NULL_TOP_M,
                n_iter=STRICT_RANDOM_NULL_ITER,
                seed=RANDOM_SEED + int(K),
            )

            if skipped is not None:
                strict_random_null_skipped.append(skipped)

            if out is not None:
                strict_random_null_rows.append(out)

strict_random_archetype_null_df = (
    pd.concat(strict_random_null_rows, ignore_index=True)
    if strict_random_null_rows else pd.DataFrame()
)

strict_random_null_skipped_df = pd.DataFrame(strict_random_null_skipped)

print("Strict random-archetype null rows:", len(strict_random_archetype_null_df))
print("Skipped cases:", len(strict_random_null_skipped_df))

display(strict_random_archetype_null_df.head())
display(strict_random_null_skipped_df.head(30))


def plot_strict_random_null_metric(null_df, metric="gini", condition="intact"):
    sub = null_df[
        (null_df["metric"] == metric) &
        (null_df["condition"] == condition)
    ].copy()

    if len(sub) == 0:
        print("No strict random null rows:", metric, condition)
        return

    fig, ax = plt.subplots(figsize=(7.5, 4.8))

    for analysis_type, ls in [("spatial", "-"), ("temporal", "--")]:
        a = sub[sub["analysis_type"] == analysis_type].sort_values("component_ratio")
        if len(a) == 0:
            continue

        ax.plot(
            a["component_ratio"],
            a["observed"],
            marker="o",
            linestyle=ls,
            linewidth=2.2,
            label=f"{analysis_type} top-decoding",
        )

        ax.plot(
            a["component_ratio"],
            a["null_mean"],
            linestyle=ls,
            linewidth=1.5,
            alpha=0.65,
            label=f"{analysis_type} random null",
        )

        ax.fill_between(
            a["component_ratio"],
            a["null_mean"] - 1.96 * a["null_std"],
            a["null_mean"] + 1.96 * a["null_std"],
            alpha=0.15,
        )

        for _, r in a.iterrows():
            if abs(r["z"]) >= 2:
                ax.text(
                    r["component_ratio"],
                    r["observed"],
                    f"K={int(r['K'])}",
                    fontsize=8,
                    ha="center",
                    va="bottom",
                )

    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel(metric)
    ax.set_title(f"{condition}: top-decoding archetypes vs random-archetype null\nstrictly excluding K <= top_m")
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    save_current_fig(f"strict_random_archetype_null_{condition}_{metric}")
    plt.show()
    plt.close()


for condition in STRICT_RANDOM_NULL_CONDITIONS:
    for metric in ["gini", "entropy_norm", "effective_n_nodes", "participation_ratio"]:
        plot_strict_random_null_metric(strict_random_archetype_null_df, metric=metric, condition=condition)


In [ ]:

# ============================================================
# MATCHED-RATIO SPATIAL VS TEMPORAL SUMMARY
# ============================================================
# Compares spatial and temporal top-decoding archetype summaries at similar
# normalized component ratios.

MATCHED_RATIO_MAX_DIFF = 0.025
MATCHED_RATIO_METHOD = "symmetric_best"
MATCHED_RATIO_CONDITIONS = ["intact", "word", "rest"]

def summarize_selected_archetypes_for_matching(localization_df, network_df, top_rank_max=TOP_N_ARCHETYPES):
    loc = localization_df[
        localization_df["rank_within_condition"] <= top_rank_max
    ].copy()

    loc_summary = (
        loc.groupby(["analysis_type", "K", "component_ratio", "condition"], as_index=False)
        .agg(
            decoding_mean=("decoding_mean", "mean"),
            gini_mean=("gini", "mean"),
            entropy_mean=("entropy_norm", "mean"),
            effective_n_nodes_mean=("effective_n_nodes", "mean"),
            participation_ratio_mean=("participation_ratio", "mean"),
        )
    )

    net = network_df[
        network_df["rank_within_condition"] <= top_rank_max
    ].copy()

    arch_net = (
        net.groupby(["analysis_type", "K", "component_ratio", "condition", "archetype"], as_index=False)
        .agg(
            max_abs_network_enrichment_z=("enrichment_z", lambda x: np.max(np.abs(x))),
            max_network_enrichment_z=("enrichment_z", "max"),
            min_network_enrichment_z=("enrichment_z", "min"),
        )
    )

    net_summary = (
        arch_net.groupby(["analysis_type", "K", "component_ratio", "condition"], as_index=False)
        .agg(
            max_abs_network_enrichment_z_mean=("max_abs_network_enrichment_z", "mean"),
            max_network_enrichment_z_mean=("max_network_enrichment_z", "mean"),
            min_network_enrichment_z_mean=("min_network_enrichment_z", "mean"),
        )
    )

    return loc_summary.merge(
        net_summary,
        on=["analysis_type", "K", "component_ratio", "condition"],
        how="left"
    )

matched_source_summary_df = summarize_selected_archetypes_for_matching(
    localization_df,
    network_df,
    top_rank_max=TOP_N_ARCHETYPES
)

display(matched_source_summary_df.head())


def build_matched_ratio_pairs(summary_df, condition, max_diff=MATCHED_RATIO_MAX_DIFF, method=MATCHED_RATIO_METHOD):
    spatial = summary_df[
        (summary_df["analysis_type"] == "spatial") &
        (summary_df["condition"] == condition)
    ].copy()

    temporal = summary_df[
        (summary_df["analysis_type"] == "temporal") &
        (summary_df["condition"] == condition)
    ].copy()

    pairs = []
    for _, s in spatial.iterrows():
        for _, t in temporal.iterrows():
            ratio_diff = abs(float(s["component_ratio"]) - float(t["component_ratio"]))
            if ratio_diff <= max_diff:
                row = {
                    "condition": condition,
                    "K_spatial": int(s["K"]),
                    "K_temporal": int(t["K"]),
                    "ratio_spatial": float(s["component_ratio"]),
                    "ratio_temporal": float(t["component_ratio"]),
                    "ratio_mid": float((s["component_ratio"] + t["component_ratio"]) / 2),
                    "ratio_diff": ratio_diff,
                }

                metric_cols = [
                    "decoding_mean",
                    "gini_mean",
                    "entropy_mean",
                    "effective_n_nodes_mean",
                    "participation_ratio_mean",
                    "max_abs_network_enrichment_z_mean",
                    "max_network_enrichment_z_mean",
                    "min_network_enrichment_z_mean",
                ]

                for col in metric_cols:
                    row[f"spatial_{col}"] = float(s[col])
                    row[f"temporal_{col}"] = float(t[col])
                    row[f"temporal_minus_spatial_{col}"] = float(t[col] - s[col])

                pairs.append(row)

    candidates = pd.DataFrame(pairs)

    if len(candidates) == 0:
        return candidates

    candidates = candidates.sort_values("ratio_diff").copy()

    if method == "spatial_to_temporal":
        return candidates.groupby(["condition", "K_spatial"], as_index=False).head(1).reset_index(drop=True)

    if method == "temporal_to_spatial":
        return candidates.groupby(["condition", "K_temporal"], as_index=False).head(1).reset_index(drop=True)

    if method == "symmetric_best":
        keep = []
        used_s = set()
        used_t = set()

        for _, r in candidates.iterrows():
            s_key = (condition, int(r["K_spatial"]))
            t_key = (condition, int(r["K_temporal"]))

            if s_key in used_s or t_key in used_t:
                continue

            keep.append(r)
            used_s.add(s_key)
            used_t.add(t_key)

        return pd.DataFrame(keep).reset_index(drop=True)

    raise ValueError("MATCHED_RATIO_METHOD must be symmetric_best, spatial_to_temporal, or temporal_to_spatial.")


matched_ratio_df = pd.concat(
    [
        build_matched_ratio_pairs(matched_source_summary_df, condition)
        for condition in MATCHED_RATIO_CONDITIONS
    ],
    ignore_index=True
)

print("Matched-ratio pairs:", len(matched_ratio_df))
display(matched_ratio_df.head(40))


def plot_matched_ratio_difference(matched_df, metric_base, condition="intact"):
    col = f"temporal_minus_spatial_{metric_base}"

    if col not in matched_df.columns:
        print("Missing column:", col)
        return

    sub = matched_df[matched_df["condition"] == condition].sort_values("ratio_mid").copy()

    if len(sub) == 0:
        print("No matched rows for", condition)
        return

    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    ax.axhline(0, color="black", linewidth=1, linestyle="--")

    ax.plot(
        sub["ratio_mid"],
        sub[col],
        marker="o",
        linewidth=2,
    )

    for _, r in sub.iterrows():
        ax.text(
            r["ratio_mid"],
            r[col],
            f"S{int(r['K_spatial'])}/T{int(r['K_temporal'])}",
            fontsize=7,
            ha="center",
            va="bottom" if r[col] >= 0 else "top",
        )

    ax.set_xlabel("Matched normalized component ratio")
    ax.set_ylabel(f"Temporal minus spatial: {metric_base}")
    ax.set_title(f"{condition}: matched-ratio temporal-spatial difference")
    plt.tight_layout()
    save_current_fig(f"matched_ratio_difference_{condition}_{metric_base}")
    plt.show()
    plt.close()


for condition in MATCHED_RATIO_CONDITIONS:
    for metric_base in [
        "decoding_mean",
        "gini_mean",
        "entropy_mean",
        "effective_n_nodes_mean",
        "participation_ratio_mean",
        "max_abs_network_enrichment_z_mean",
    ]:
        plot_matched_ratio_difference(matched_ratio_df, metric_base, condition=condition)


In [ ]:

# ============================================================
# SAVE REFINED NULL AND MATCHED-RATIO TABLES
# ============================================================

strict_null_path = FIG_DIR / f"strict_random_archetype_null_{FIT_SCOPE}.csv"
strict_skip_path = FIG_DIR / f"strict_random_archetype_null_skipped_{FIT_SCOPE}.csv"
matched_source_path = FIG_DIR / f"matched_ratio_source_summary_{FIT_SCOPE}.csv"
matched_ratio_path = FIG_DIR / f"matched_ratio_spatial_temporal_summary_{FIT_SCOPE}.csv"

strict_random_archetype_null_df.to_csv(strict_null_path, index=False)
strict_random_null_skipped_df.to_csv(strict_skip_path, index=False)
matched_source_summary_df.to_csv(matched_source_path, index=False)
matched_ratio_df.to_csv(matched_ratio_path, index=False)

print("Saved:", strict_null_path)
print("Saved:", strict_skip_path)
print("Saved:", matched_source_path)
print("Saved:", matched_ratio_path)

display(matched_ratio_df.head(30))
display(strict_random_archetype_null_df.head(30))
